# NTB-00: Database Setup & Schema Design

## Objective
Load the raw Telco Churn CSV, design a normalized SQLite schema, and create a queryable database that will power all downstream analysis.

## Why a Database Instead of Just CSV?
- Demonstrates SQL proficiency on the CV (critical for Algerian DS roles)
- Enables complex business queries that pandas does awkwardly
- Power BI dashboard in NTB-14 will connect directly to this database
- Mirrors real-world data architecture where data lives in databases, not files

## Schema Design
We split the flat CSV into three logical tables:
- `customers`: demographic and account info (stable attributes)
- `services`: which services each customer subscribes to (the product layer)
- `billing`: financial info and churn outcome (the business layer)

This normalization isn't strictly necessary for ML, but it shows we think about data modeling — a skill recruiters test for.

## 1. Setup: Imports and Path Resolution

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import os

# Resolve the project root from either the notebook or project working directory
BASE_DIR = Path(os.getcwd())
if not (BASE_DIR / "data" / "raw").exists():
    BASE_DIR = BASE_DIR.parent

DATA_RAW = BASE_DIR / "data" / "raw"
DATA_DB = BASE_DIR / "data" / "database"

# Ensure directories exist
DATA_DB.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DB / "telco_churn.db"
RAW_CSV = DATA_RAW / "Telco-Customer-Churn.csv"

print(f"Project root: {BASE_DIR}")
print(f"Raw data:     {RAW_CSV}")
print(f"Database:     {DB_PATH}")
print(f"Raw CSV exists: {RAW_CSV.exists()}")

Project root: c:\Users\Prakhar kapoor\Desktop\Telecom-Churn-Prediction-Analytics
Raw data:     c:\Users\Prakhar kapoor\Desktop\Telecom-Churn-Prediction-Analytics\data\raw\Telco-Customer-Churn.csv
Database:     c:\Users\Prakhar kapoor\Desktop\Telecom-Churn-Prediction-Analytics\data\database\telco_churn.db
Raw CSV exists: True


## 2. Load and Inspect Raw Data

Expected: 7043 rows × 21 columns. Columns cover demographics (gender, SeniorCitizen, Partner, Dependents), services (PhoneService, InternetService, OnlineSecurity, etc.), billing (Contract, PaymentMethod, MonthlyCharges, TotalCharges), and the target (Churn).

In [2]:
# Load raw CSV
df = pd.read_csv(RAW_CSV)

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nDtypes:\n{df.dtypes}")
df.head()

Shape: (7043, 21)

Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Dtypes:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Data Quality Check

The famous quirk of this dataset: `TotalCharges` is stored as a string, and 11 rows contain just whitespace instead of a number. This is exactly the kind of real-world data issue interviewers ask about — being able to explain why and how you handled it scores points.

In [3]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# TotalCharges is stored as string in this dataset — known quirk
print(f"\nTotalCharges dtype: {df['TotalCharges'].dtype}")
print(f"Sample TotalCharges values: {df['TotalCharges'].head().tolist()}")

# Check for whitespace strings (the actual missing value pattern in this dataset)
whitespace_mask = df['TotalCharges'].str.strip() == ''
print(f"\nRows with empty TotalCharges: {whitespace_mask.sum()}")

Missing values per column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

TotalCharges dtype: str
Sample TotalCharges values: ['29.85', '1889.5', '108.15', '1840.75', '151.65']

Rows with empty TotalCharges: 11


In [4]:
# Convert TotalCharges to numeric, coercing whitespace to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f"Missing TotalCharges after conversion: {df['TotalCharges'].isnull().sum()}")

# Investigate: these 11 rows are likely brand-new customers (tenure = 0)
print(f"\nTenure for rows with missing TotalCharges:")
print(df[df['TotalCharges'].isnull()]['tenure'].value_counts())

Missing TotalCharges after conversion: 11

Tenure for rows with missing TotalCharges:
tenure
0    11
Name: count, dtype: int64


**Finding:** The 11 missing `TotalCharges` all have `tenure = 0`. These are brand-new customers who haven't been billed yet. We store the data as-is in the database to preserve the original structure — imputation happens in NTB-02 (feature engineering).

## 4. Normalize Into Three Logical Tables

We split the flat CSV into three tables with `customer_id` as the foreign key. We also rename columns to `snake_case` (Python/SQL convention) and convert `Churn` from Yes/No to 1/0 for ML readiness.

In [5]:
# Table 1: customers (demographic + account stable info)
customers_cols = [
    'customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure'
]
customers_df = df[customers_cols].copy()
customers_df.columns = ['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months']

# Table 2: services (what the customer subscribes to)
services_cols = [
    'customerID', 'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies'
]
services_df = df[services_cols].copy()
services_df.columns = [
    'customer_id', 'phone_service', 'multiple_lines', 'internet_service',
    'online_security', 'online_backup', 'device_protection', 'tech_support',
    'streaming_tv', 'streaming_movies'
]

# Table 3: billing (financial + outcome)
billing_cols = [
    'customerID', 'Contract', 'PaperlessBilling', 'PaymentMethod',
    'MonthlyCharges', 'TotalCharges', 'Churn'
]
billing_df = df[billing_cols].copy()
billing_df.columns = [
    'customer_id', 'contract_type', 'paperless_billing', 'payment_method',
    'monthly_charges', 'total_charges', 'churn'
]

# Convert Churn from Yes/No to 1/0 for ML readiness
billing_df['churn'] = (billing_df['churn'] == 'Yes').astype(int)

print(f"customers: {customers_df.shape}")
print(f"services:  {services_df.shape}")
print(f"billing:   {billing_df.shape}")

print(f"\nChurn distribution:\n{billing_df['churn'].value_counts()}")

customers: (7043, 6)
services:  (7043, 10)
billing:   (7043, 7)

Churn distribution:
churn
0    5174
1    1869
Name: count, dtype: int64


## 5. Create the SQLite Schema

Three tables with proper types, primary keys, foreign keys, and a CHECK constraint on `churn`. Indexes on the columns we'll join and filter most often.

**Why drop tables first?** Re-running this notebook should be idempotent — running it twice shouldn't create duplicates or errors.

In [6]:
# Connect and create database
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Drop tables if they exist (lets us re-run this notebook safely)
cursor.execute("DROP TABLE IF EXISTS billing")
cursor.execute("DROP TABLE IF EXISTS services")
cursor.execute("DROP TABLE IF EXISTS customers")

# Create tables with proper types and constraints
cursor.execute("""
CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    gender TEXT NOT NULL,
    senior_citizen INTEGER NOT NULL,
    partner TEXT NOT NULL,
    dependents TEXT NOT NULL,
    tenure_months INTEGER NOT NULL
)
""")

cursor.execute("""
CREATE TABLE services (
    customer_id TEXT PRIMARY KEY,
    phone_service TEXT NOT NULL,
    multiple_lines TEXT NOT NULL,
    internet_service TEXT NOT NULL,
    online_security TEXT NOT NULL,
    online_backup TEXT NOT NULL,
    device_protection TEXT NOT NULL,
    tech_support TEXT NOT NULL,
    streaming_tv TEXT NOT NULL,
    streaming_movies TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

cursor.execute("""
CREATE TABLE billing (
    customer_id TEXT PRIMARY KEY,
    contract_type TEXT NOT NULL,
    paperless_billing TEXT NOT NULL,
    payment_method TEXT NOT NULL,
    monthly_charges REAL NOT NULL,
    total_charges REAL,
    churn INTEGER NOT NULL CHECK (churn IN (0, 1)),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

# Create indexes for the joins and filters we'll use most
cursor.execute("CREATE INDEX idx_billing_churn ON billing(churn)")
cursor.execute("CREATE INDEX idx_billing_contract ON billing(contract_type)")
cursor.execute("CREATE INDEX idx_customers_tenure ON customers(tenure_months)")

conn.commit()
print("Schema created successfully.")

Schema created successfully.


## 6. Load Data Into the Database

In [7]:
# Use pandas .to_sql() with if_exists='append' since we already created the schema
customers_df.to_sql('customers', conn, if_exists='append', index=False)
services_df.to_sql('services', conn, if_exists='append', index=False)
billing_df.to_sql('billing', conn, if_exists='append', index=False)

# Verify row counts
for table in ['customers', 'services', 'billing']:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")

conn.commit()

customers: 7043 rows
services: 7043 rows
billing: 7043 rows


## 7. Sanity Check: Overall Churn Rate

Expected churn rate for this dataset: **~26.5%**. If you see this number, the data loaded correctly.

In [8]:
# Overall churn rate
query = """
SELECT 
    COUNT(*) as total_customers,
    SUM(churn) as churned_customers,
    ROUND(100.0 * SUM(churn) / COUNT(*), 2) as churn_rate_pct
FROM billing
"""
result = pd.read_sql_query(query, conn)
print("Overall churn metrics:")
print(result)

Overall churn metrics:
   total_customers  churned_customers  churn_rate_pct
0             7043               1869           26.54


## 8. The Money Query: Churn by Contract Type × Tenure Bucket

This single query reveals the **most important pattern in the dataset**: month-to-month customers in their first year have catastrophically high churn rates (>50%), while two-year contract customers churn at <3% regardless of tenure.

This is the kind of query that takes a churn analysis from "ran sklearn on a CSV" to "understands the business." We'll build on this finding heavily in NTB-01.

In [9]:
# Churn rate by contract type and tenure bucket
query = """
SELECT 
    b.contract_type,
    CASE 
        WHEN c.tenure_months <= 12 THEN '0-12 months'
        WHEN c.tenure_months <= 24 THEN '13-24 months'
        WHEN c.tenure_months <= 48 THEN '25-48 months'
        ELSE '49+ months'
    END AS tenure_bucket,
    COUNT(*) as customer_count,
    SUM(b.churn) as churned,
    ROUND(100.0 * SUM(b.churn) / COUNT(*), 2) as churn_rate_pct,
    ROUND(AVG(b.monthly_charges), 2) as avg_monthly_charges
FROM billing b
JOIN customers c ON b.customer_id = c.customer_id
GROUP BY b.contract_type, tenure_bucket
ORDER BY b.contract_type, tenure_bucket
"""
result = pd.read_sql_query(query, conn)
print("Churn analysis by contract type and tenure:")
print(result.to_string(index=False))

Churn analysis by contract type and tenure:
 contract_type tenure_bucket  customer_count  churned  churn_rate_pct  avg_monthly_charges
Month-to-month   0-12 months            1994     1024           51.35                58.22
Month-to-month  13-24 months             737      278           37.72                69.31
Month-to-month  25-48 months             802      264           32.92                75.94
Month-to-month    49+ months             342       89           26.02                85.45
      One year   0-12 months             124       13           10.48                35.80
      One year  13-24 months             197       16            8.12                44.88
      One year  25-48 months             518       55           10.62                61.96
      One year    49+ months             634       82           12.93                79.56
      Two year   0-12 months              68        0            0.00                30.95
      Two year  13-24 months              90  

## 9. Clean Shutdown

Always close database connections. SQLite handles this gracefully but explicit cleanup is good practice.

In [10]:
conn.close()
print(f"Database closed. File saved at: {DB_PATH}")
print(f"File size: {DB_PATH.stat().st_size / 1024:.1f} KB")

Database closed. File saved at: c:\Users\Prakhar kapoor\Desktop\Telecom-Churn-Prediction-Analytics\data\database\telco_churn.db
File size: 1992.0 KB


## ✅ What You Have Now

A working SQLite database at `data/database/telco_churn.db` with:
- 3 normalized tables (`customers`, `services`, `billing`)
- 7043 customers loaded across each table
- Proper primary keys, foreign keys, and indexes
- A CHECK constraint enforcing churn ∈ {0, 1}

You can query it from:
- Any Python notebook (`sqlite3` or `sqlalchemy`)
- **DBeaver** (free SQL GUI — recommended for browsing)
- **Power BI** (we'll connect it in NTB-14)

## Next: NTB-01 — Exploratory Data Analysis with Business Questions

We'll write 8–10 SQL queries that answer real business questions (revenue at risk, contract cliff, payment method risk profile, etc.) and produce the first set of plots for your README.